# Local OT Potential Test

Use this notebook for local smoke tests and medium-scale exploratory runs. The command-line scripts are intended for cluster and Slurm usage.

In [8]:
import matplotlib.pyplot as plt
import numpy as np

from otexp.experiment import run_ball_experiment

## Parameters

Start with the tiny smoke-test values below. For a medium local test, try `n_target = (256,)`, `B = 3`, `n_source = 4096`, and `max_iter = 80`.

In [11]:
d = 2
n_target = (16, 32)
B = 2
n_source = 512
seed = 2026
max_iter = 30
chunk_size = 512
overwrite = False
outdir = "notebook/local_results"

## Run Experiment

In [12]:
df, slope_loss, slope_beta = run_ball_experiment(
    d=d,
    n_target=n_target,
    B=B,
    n_source=n_source,
    seed=seed,
    outdir = outdir,
    max_iter=max_iter,
    chunk_size=chunk_size,
    overwrite=overwrite,
    save_results=False,
)

df

TypeError: run_ball_experiment() got an unexpected keyword argument 'save_results'

In [ ]:
print(f"Empirical log-log slope for mean loss: {slope_loss:.3f}")
print(f"Log-log slope of beta(n,{d}): {slope_beta:.3f}")

## Plot Results

In [ ]:
valid = df["mean_loss"].notna() & (df["mean_loss"] > 0)
slope = np.nan
intercept = np.nan
if valid.sum() >= 2:
    slope, intercept = np.polyfit(
        np.log(df.loc[valid, "n"]),
        np.log(df.loc[valid, "mean_loss"]),
        deg=1,
    )

beta_valid = df["beta"].notna() & (df["beta"] > 0) & valid
rate_slope = np.nan
rate_intercept = np.nan
if beta_valid.sum() >= 2:
    rate_slope, rate_intercept = np.polyfit(
        np.log(df.loc[beta_valid, "n"]),
        np.log(df.loc[beta_valid, "beta"]),
        deg=1,
    )
    scale = np.exp(
        np.mean(
            np.log(df.loc[beta_valid, "mean_loss"])
            - np.log(df.loc[beta_valid, "beta"])
        )
    )
else:
    scale = np.nan

fig, ax = plt.subplots(figsize=(5.5, 3.8))
ax.errorbar(
    df["n"],
    df["mean_loss"],
    yerr=1.96 * df["se_loss"],
    fmt="o-",
    color="black",
    capsize=3,
    label="mean loss",
)

if valid.sum() >= 2:
    n_line = np.logspace(
        np.log10(df.loc[valid, "n"].min()),
        np.log10(df.loc[valid, "n"].max()),
        100,
    )
    ax.plot(
        n_line,
        np.exp(intercept) * n_line**slope,
        color="tab:blue",
        label=f"fit slope = {slope:.2f}",
    )

if beta_valid.sum() >= 2:
    n_rate = df.loc[beta_valid, "n"].to_numpy()
    beta_rate = df.loc[beta_valid, "beta"].to_numpy()
    order = np.argsort(n_rate)
    ax.plot(
        n_rate[order],
        scale * beta_rate[order],
        "--",
        color="tab:red",
        label=f"scaled rate slope = {rate_slope:.2f}",
    )

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("n")
ax.set_ylabel("loss")
ax.set_title(f"d = {d}")
ax.legend()
fig.tight_layout()

In [ ]:
print(f"Empirical log-log slope: {slope:.3f}")
print(f"Theoretical rate log-log slope over this grid: {rate_slope:.3f}")